# Lab 2 on the RTX 4090

This runs the repaired RAG and image-agent experiments in order and keeps their executed notebooks. It reuses your completed Part 2 training, 120 generated images and measured metrics. It does not repeat training or count the three incomplete RAG answers as finished work.

Before Run All, connect Colab to the local runtime on the RTX 4090 PC, or open this file in Jupyter on that PC. A hosted Colab GPU is a different machine. Use Python 3.10 through 3.12 if available; setup creates a separate pinned environment if needed. Git, internet access and the NVIDIA driver must work. Allow roughly 40 GB free disk for a cold installation and at least 8 GiB available host RAM. Close other GPU notebooks, plug in power and disable sleep.

Keep your original `Part2_results_*.zip` on the GPU machine. Its files are not restored just by opening GitHub. The next cell can locate that archive in Downloads, the repository, `/content` or `/app`. If it is elsewhere, set its runtime path below. A Docker kernel needs a mounted folder or an uploaded ZIP, not a Windows path the container cannot see.

Leave `LAB2_SMOKE_ONLY` unset for real computation. Setting it to `1` is a developer test that skips all model-backed work. Human ratings, reference review, team details, permissions, the demo and final external submissions still need real people. A 24-GB 4090 run is not evidence of execution on the rubric's 16-GB card.

In [ ]:
import os

# Only change this if the original ZIP is not found automatically.
# Example on Windows: r"C:\Users\YOUR_NAME\Downloads\Part2_results_20261106_025010.zip"
# Example in a Linux container with the ZIP uploaded: "/app/Part2_results_20261106_025010.zip"
PART2_RESULTS_ZIP = os.environ.get("LAB2_PART2_RESULTS_ZIP", "")

# Optional: set LAB2_ROOT before setup if you use a different existing checkout.
# Never put a Hugging Face token or password in this cell.
os.environ.setdefault("LAB2_UPDATE_REPOSITORY", "1")

## Set up the isolated runtime

This checks the visible GPU before installing packages. A patched old checkout is preserved; a clean sibling checkout is used instead. All following scripts run in the pinned environment, not whichever packages happen to be in the Colab kernel.

In [ ]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
os.environ["LAB2_DEVICE"] = "cpu" if SMOKE_ONLY else "cuda"
if not SMOKE_ONLY:
    try:
        gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True, timeout=15)
    except (OSError, subprocess.SubprocessError) as error:
        raise RuntimeError("Connect this notebook to your RTX 4090 kernel first. A normal hosted Colab kernel is not your PC GPU.") from error
    if "4090" not in gpu_names:
        raise RuntimeError("No RTX 4090 is visible to this kernel. Correct the runtime before downloading models.")
    print("Visible GPU(s):", gpu_names.strip())
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)

if not (ROOT / "tools/rtx4090_workflow.py").is_file():
    old_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-rtx4090-v1")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/rtx4090_workflow.py").is_file():
        branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        if branch == "main" and not dirty:
            subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    if not (ROOT / "tools/rtx4090_workflow.py").is_file():
        raise RuntimeError(f"The safe sibling checkout at {ROOT} is outdated. Your original files at {old_root} were preserved.")
    print("Using schema-constrained code at", ROOT)
    print("The earlier checkout, local edits and results are unchanged at", old_root)

subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")

# Permit first-time downloads during preflight, even after restarting this notebook.
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
# Monitor actual device memory in each subprocess. A larger GPU is not relabeled as a T4.
_run_stage = run_script
def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    _run_stage("tools/monitored_run.py", "--record", record, "--", PYTHON, ROOT / relative, *arguments)


## Test the code and check the actual hardware

The offline tests cover response structure, image handling, recovery and ZIP downloads. The CUDA check then runs on the real 4090. These checks are separate from model-quality evaluation.

In [ ]:
for test in ["test_lab_completion.py", "test_structured_responses.py", "test_constrained_decoding.py",
             "test_overnight_resume.py", "test_part2_submission.py", "test_result_download.py",
             "test_rtx4090_workflow.py"]:
    run_script("tools/" + test)
if SMOKE_ONLY:
    print("SKIPPED: actual RTX 4090 verification. CPU smoke is not GPU evidence.")
else:
    run_script("tools/rtx4090_workflow.py", "profile")

## Restore the completed Part 2 results

If all original files are already here, they are reused. Otherwise, the original ZIP is imported with backups. Both adapters must match their recorded uploaded hashes. All 120 image combinations, image shapes, saved training settings, quality metrics and memorization figures are checked. A missing archive stops here before expensive inference.

The selected adapter is copied from this verified local export, so no private Hub token is needed for that copy. This does not verify grader access to the private Hub repositories. Dataset metadata alone is not enough: missing source images are downloaded again using the original pinned dataset and split.

In [ ]:
DATA = Path(os.environ.get("LAB2_DATA_DIR", str(ROOT / "part1_rag/data"))).expanduser().resolve()
QUESTIONS = Path(os.environ.get("QUESTIONS_PATH", str(DATA / "Questions.csv"))).expanduser().resolve()
if not SMOKE_ONLY:
    if not QUESTIONS.is_file():
        raise FileNotFoundError("Questions.csv is missing. Attach the supplied Part 1 data or set LAB2_DATA_DIR and QUESTIONS_PATH.")
    arguments = ["reuse-part2"]
    if PART2_RESULTS_ZIP:
        arguments += ["--archive", PART2_RESULTS_ZIP]
    run_script("tools/rtx4090_workflow.py", *arguments)
    run_script("part2_diffusion/src/prepare_data.py", "--output-dir", ROOT / "part2_diffusion/data/processed")
    run_script("part3_agents/src/prepare_evaluation.py", "--download-adapter")
else:
    print("SKIPPED: original Part 2 import, dataset download and adapter preparation.")

## Run the real models before leaving the PC

These checks load the real embedding, visual-language, image-editing and style-generation paths, including the original domain adapter. They run sequentially and populate the model cache. Keep the notebook connected and confirm `BOTH REAL GPU PREFLIGHTS PASSED` before leaving the PC. Two-step diagnostic images are not final evaluation samples.

In [ ]:
if SMOKE_ONLY:
    print("SKIPPED: real GPU model preflights. Do not leave a GPU job based only on this smoke output.")
else:
    run_script("tools/overnight_preflight.py", "--part", "part1_rag", "--data-dir", DATA)
    run_script("tools/overnight_preflight.py", "--part", "part3_agents")
    print("BOTH REAL GPU PREFLIGHTS PASSED. The sequential full run starts in the next cell.")

## Run Parts 1 and 3 and save the executed notebooks

Part 1 generates and validates the supplied question CSV, then evaluates the dense baseline and final system. Part 3 runs 15 scenarios with and without critique and ten image-aware safety cases for each. There are at most two revisions per request. Models run one at a time.

Completed unchanged items resume after an interruption. A changed source, input or GPU creates a separate checkpoint namespace. Model errors are saved rather than turned into fake answers or successful safety refusals. Executed notebook copies save after every cell under `output/executed`. Keep the GPU runtime and local server running.

In [ ]:
run_script("tools/overnight_runner.py", *(["--smoke-only"] if SMOKE_ONLY else []))
print("Executed notebook folder:", ROOT / "output/executed")
print("Progress record:", ROOT / "runtime_checks/overnight_progress.json")

## Check the saved results

This checks execution separately from answer quality and submission readiness. A script exiting normally does not hide model errors in its saved traces. Expected corrupt-image refusals remain safety tests, not runtime crashes. Real ratings stay blank until the raters fill them in.

The reports refresh from saved measurements in the inner notebooks. They stay drafts while real reviews and external requirements are pending. If this cell fails, keep its diagnostic and run the ZIP cell below; do not delete completed work or edit failures out of the notebook.

In [ ]:
run_script("tools/rtx4090_workflow.py", "completion", *(["--smoke-only"] if SMOKE_ONLY else []))
run_script("tools/prepare_rating_forms.py")
run_script("tools/final_submission_audit.py")
print("Readiness audit:", ROOT / "runtime_checks/final_submission_audit.json")
print("The next cell exports all available results. It does not submit to Canvas or Kaggle.")

## Download one ZIP with all available lab results

Run this cell after the overnight work. It bundles all available results for Parts 1, 2 and 3, executed notebooks, source/configuration, reports, adapters, generated images and diagnostic logs. Base-model downloads, environments, original datasets and old archives are excluded. Missing files are listed honestly in `BUNDLE_MANIFEST.json`.

The ZIP transfers through the Colab notebook connection, including a local kernel, without a localhost hyperlink. A native Windows/Mac runtime also copies it into its accessible Downloads folder. Browser location follows your download settings. Keep the kernel connected until transfer finishes. If the run fails, you can execute only this export cell to send the partial evidence; no GPU work is repeated.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, importlib.util

if "ROOT" not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents,
        Path(os.environ.get("LAB2_ROOT", str(Path.home() / "DATA266/Lab-2"))),
        Path("/content/Lab-2"), Path("/kaggle/working/Lab-2")]
    ROOT = next((p.resolve() for p in candidates if (p / "tools/export_lab_results.py").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Set LAB2_ROOT to your existing Lab-2 folder. Do not rerun training just to export files.")

# This script only collects saved files. It does not install packages, train, or generate.
subprocess.run([sys.executable, str(ROOT / "tools/export_lab_results.py"), "--all"], cwd=ROOT, check=True)
bundle = json.loads((ROOT / "output/exports/latest_all_results.json").read_text())
print("Upload this ZIP in our chat:", bundle["archive"])
print("ZIP SHA256:", bundle["sha256"])
if globals().get("SMOKE_ONLY", False):
    print("CPU smoke: bundle verified, browser download intentionally not triggered. These are not new GPU results.")
else:
    spec = importlib.util.spec_from_file_location("lab2_download_results", ROOT / "tools/download_results.py")
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloader.offer_download(bundle["archive"], auto_start=True)